# Assignment 3: Regularization and Model Stability
**ECON 417 Business Forecasting — Southern Illinois University Edwardsville**

**Student Version**

---
### Instructions
This assignment covers Lecture 3: Regularization and Model Stability. Topics include coefficient instability, multicollinearity, Ridge regression, Lasso regression, λ tuning, and the StandardScaler best practice.

- **Q3 and Q7** are coding questions — complete the code cells.
- All other questions are written responses — type your answers in the answer cells.
- Always use the **validation set** to choose λ — never the test set.
- **StandardScaler** must be fitted on training data only.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# ── Load data (yfinance with synthetic fallback) ───────────────────────────
try:
    import yfinance as yf
    _raw = yf.download(['^GSPC','^VIX','MSFT','AAPL','JPM','XOM'],
                       start='2015-01-01', end='2024-12-31', auto_adjust=True)
    _close = _raw['Close']
    df = pd.DataFrame({
        'sp500':    _close['^GSPC'].pct_change(),
        'vix_chg':  _close['^VIX'].diff(),
        'lag_msft': _close['MSFT'].pct_change().shift(1),
        'lag_aapl': _close['AAPL'].pct_change().shift(1),
        'lag_jpm':  _close['JPM'].pct_change().shift(1),
        'lag_xom':  _close['XOM'].pct_change().shift(1),
    }).dropna()
    print("✓ Live data loaded from yfinance.")
except Exception:
    print("yfinance unavailable – using synthetic data.")
    np.random.seed(42)
    dates = pd.date_range('2015-01-02', '2024-12-31', freq='B')
    n = len(dates)
    sp500 = np.random.randn(n) * 0.01
    msft  = sp500 * 1.10 + np.random.randn(n) * 0.008
    aapl  = sp500 * 1.05 + np.random.randn(n) * 0.009
    jpm   = sp500 * 0.90 + np.random.randn(n) * 0.010
    xom   = sp500 * 0.70 + np.random.randn(n) * 0.010
    vix   = -sp500 * 100 + np.random.randn(n) * 2
    df = pd.DataFrame({
        'sp500':    sp500,
        'vix_chg':  np.concatenate([[np.nan], np.diff(vix)]),
        'lag_msft': np.concatenate([[np.nan], msft[:-1]]),
        'lag_aapl': np.concatenate([[np.nan], aapl[:-1]]),
        'lag_jpm':  np.concatenate([[np.nan], jpm[:-1]]),
        'lag_xom':  np.concatenate([[np.nan], xom[:-1]]),
    }, index=dates).dropna()

# ── 60 / 20 / 20 chronological split ─────────────────────────────────────
n  = len(df)
t1 = int(n * 0.60)
t2 = int(n * 0.80)

df_train = df.iloc[:t1]
df_val   = df.iloc[t1:t2]
df_test  = df.iloc[t2:]

features = ['vix_chg', 'lag_msft', 'lag_aapl', 'lag_jpm', 'lag_xom']
target   = 'sp500'

X_train_raw = df_train[features].values
X_val_raw   = df_val[features].values
X_test_raw  = df_test[features].values
y_train     = df_train[target].values
y_val       = df_val[target].values
y_test      = df_test[target].values

# ── Standardize (fit on training data only) ───────────────────────────────
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val   = scaler.transform(X_val_raw)
X_test  = scaler.transform(X_test_raw)

print(f"Train: {len(df_train)} obs | Val: {len(df_val)} obs | Test: {len(df_test)} obs")
print(f"Features: {features}")


## Q1. Coefficient Instability (Written)

What is **coefficient instability** in OLS regression? Why is it a problem for forecasting?

**Your Answer:**

_Type your answer here._

## Q2. Multicollinearity (Written)

Define **multicollinearity**. In the S&P 500 dataset, we use lagged daily returns of MSFT, AAPL, JPM, and XOM as predictors. Why might this cause multicollinearity, and what effect does it have on OLS coefficient estimates?

**Your Answer:**

_Type your answer here._

## Q3. ⭐ Coding: Rolling OLS Coefficient Stability

Fit an OLS model on rolling 126-trading-day windows (≈ 6 months). At each step, standardize the feature window before fitting. Collect the OLS coefficient for each feature at each window, then plot all five coefficient time series in subplots.

**What to observe:** Do the coefficients remain stable? Do any change sign?

In [ ]:
# ── Q3: Rolling OLS Coefficient Stability ────────────────────────────────
# TODO: For each rolling window of 126 trading days, fit OLS on standardized
# features and record each coefficient. Then plot all 5 coefficient series.

window = 126
roll_dates = []
roll_coefs = {f: [] for f in features}

for i in range(window, len(df)):
    win = df.iloc[i - window : i]
    # YOUR CODE HERE: standardize win[features], fit LinearRegression, append coefs
    pass

if not roll_dates:
    print("⚠️  Complete the loop above, then re-run.")
else:
    fig, axes = plt.subplots(len(features), 1, figsize=(13, 10), sharex=True)
    for ax, f in zip(axes, features):
        ax.plot(roll_dates, roll_coefs[f], linewidth=0.9)
        ax.axhline(0, color='gray', linestyle='--', linewidth=0.6)
        ax.set_ylabel(f, fontsize=9)
    fig.suptitle('Rolling OLS Coefficient Stability (126-day window)', fontsize=13, fontweight='bold')
    plt.xlabel('Date')
    plt.tight_layout()
    plt.show()


## Q4. Ridge Regression (Written)

Write the Ridge regression objective function. What role does the penalty parameter **λ** play, and what happens as λ → 0 and as λ → ∞?

**Your Answer:**

_Type your answer here._

## Q5. Lasso vs. Ridge (Written)

Explain the Lasso penalty. Why can Lasso produce **exactly zero** coefficients, while Ridge cannot?

**Your Answer:**

_Type your answer here._

## Q6. Bias-Variance Tradeoff in Regularization (Written)

How does increasing λ affect model **bias** and **variance**? Explain the tradeoff and why we seek an optimal λ.

**Your Answer:**

_Type your answer here._

## Q7. ⭐ Coding: Ridge and Lasso λ Tuning

Use `lambdas = np.logspace(-4, 3, 100)`. For each λ:
1. Fit Ridge and Lasso on `X_train` / `y_train`.
2. Compute validation RMSE for each.

Then:
- Plot val RMSE vs log(λ) for both Ridge and Lasso. Mark the best λ.
- Fit OLS, best Ridge, and best Lasso on the training set.
- Print a table of Train / Val / Test RMSE for all three models.
- Print Lasso coefficients for each feature — note which are zeroed out.

In [ ]:
# ── Q7: Ridge and Lasso λ Tuning on Validation Set ───────────────────────
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

lambdas = np.logspace(-4, 3, 100)
ridge_val_rmse = []
lasso_val_rmse = []

# TODO: Loop over lambdas. For each lambda, fit Ridge and Lasso on X_train/y_train,
# predict on X_val, compute RMSE and append to the respective lists.
for lam in lambdas:
    # YOUR CODE HERE
    pass

if not ridge_val_rmse:
    print("⚠️  Complete the tuning loop above, then re-run.")
else:
    best_ridge_lam = lambdas[np.argmin(ridge_val_rmse)]
    best_lasso_lam = lambdas[np.argmin(lasso_val_rmse)]

    # TODO: Plot val RMSE vs log(lambda) for both Ridge and Lasso.
    # Mark the best lambda with a vertical dashed line.
    # YOUR CODE HERE

    # TODO: Fit OLS, best Ridge, and best Lasso on X_train.
    # Print a table of Train/Val/Test RMSE for all three models.
    # YOUR CODE HERE

    # TODO: Print the Lasso coefficients for each feature.
    # Note which ones are zeroed out.
    # YOUR CODE HERE


## Q8. StandardScaler Best Practice (Written)

Why must `StandardScaler` be **fitted only on the training set** before transforming the validation and test sets? What goes wrong if we fit it on all data at once?

**Your Answer:**

_Type your answer here._

## Q9. Ridge vs. Lasso: When to Use Which (Written)

In practice, when would you **prefer Ridge** over Lasso, and when would **Lasso** be preferred?

**Your Answer:**

_Type your answer here._

## Q10. Interpreting Lasso Output (Written)

After fitting a Lasso model, you find that the coefficients for `lag_aapl` and `lag_xom` are exactly zero, while `lag_msft` and `vix_chg` remain nonzero. How do you interpret this result? Is this sufficient evidence that AAPL and XOM returns do not predict the S&P 500?

**Your Answer:**

_Type your answer here._

---
**End of Assignment 3**
*ECON 417 Business Forecasting · Southern Illinois University Edwardsville*